1. Environment Setup & Imports
2. Data Acquisition & Corpus Loading
3. Text Preprocessing & Tokenisation
4. Sequence Generation & Dataset Construction
5. Model Architecture (Bi-LSTM + MHA)
6. Training with Callbacks
7. Training Visualisations
8. Test Set Evaluation
9. Text Generation / Inference Demo
10. Analysis & Conclusion


## 1 - Environment Setup & Imports

In [ ]:
# Test Code for PyTorch and CUDA setup (Not Needed for the main project, but useful for debugging)
import time
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Matrix size
N = 10000

# -------------------------
# CPU Benchmark
# -------------------------
print("\n--- CPU Benchmark ---")
x_cpu = torch.randn(N, N)
y_cpu = torch.randn(N, N)

start = time.time()
z_cpu = x_cpu @ y_cpu
end = time.time()

print(f"CPU Time: {end - start:.4f} seconds")
print("z_cpu shape:", z_cpu.shape)
print("z_cpu device:", z_cpu.device)

# -------------------------
# GPU Benchmark (only if CUDA works)
# -------------------------
if torch.cuda.is_available():
    print("\n--- GPU Benchmark ---")
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

    # Create tensors directly on GPU
    x_gpu = torch.randn(N, N, device="cuda")
    y_gpu = torch.randn(N, N, device="cuda")

    # 1) Warm-up run (important!)
    # This removes first-time overhead like kernel compilation, caching, etc.
    _ = x_gpu @ y_gpu
    torch.cuda.synchronize()

    # 2) Actual timed run
    start = time.time()

    z_gpu = x_gpu @ y_gpu
    torch.cuda.synchronize()  # wait for GPU to finish

    end = time.time()

    print(f"GPU Time: {end - start:.4f} seconds")
    print("z_gpu shape:", z_gpu.shape)
    print("z_gpu device:", z_gpu.device)

else:
    print("\nCUDA not detected. GPU benchmark skipped.")


In [ ]:
# Standard library
import os
import re
import json
import random
import string
import pickle
import warnings
warnings.filterwarnings('ignore')

# Numerical / Data
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
# https://matplotlib.org/stable/users/explain/customizing.html
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F7F4',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
PALETTE = {'train': '#534AB7', 'val': '#D85A30', 'test': '#1D9E75'}

# NLP
import nltk
# https://www.gutenberg.org/
nltk.download('gutenberg', quiet=True)
nltk.download('punkt',     quiet=True)
from nltk.corpus import gutenberg

# ML / Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks as keras_callbacks
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import top_k_accuracy_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# Todo: GPU not getting detected. Fix this before running the main project. For now, we will proceed with CPU training, but it will be much slower.
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}  (CPU training will be used)')
print('All imports successful.')

In [ ]:
# Todo: Remove after debuggibg. Additional GPU check using TensorFlow (since PyTorch check was done earlier)
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPU devices:", tf.config.list_physical_devices('GPU'))


---
## 2 — Data Acquisition & Corpus Loading

In [ ]:
# Load Shakespeare plays from NLTK Gutenberg corpus
# Todo :: 3 Shakespeare plays (Todo: Add more texts if needed)
# https://www.gutenberg.org/cache/epub/100/pg100.txt
shakespeare_files = [f for f in gutenberg.fileids() if 'shakespeare' in f]
print('Shakespeare files available in NLTK Gutenberg:')

raw_texts = []
for fname in shakespeare_files:
    raw = gutenberg.raw(fname)
    raw_texts.append(raw)
    print(f'  {fname:<30} →  {len(raw):,} chars')

# Combine all three plays into one corpus string
CORPUS_RAW = '\n\n'.join(raw_texts)
print(f'\nCombined corpus length : {len(CORPUS_RAW):,} characters')
print('First 500 characters:\n', CORPUS_RAW[:500])

---
## 3 — Text Preprocessing & Tokenisation

In [ ]:
# Preprocessing function
def clean_text(text: str) -> str:
    """Normalise Shakespeare text for language modelling."""
    text = text.lower()
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)    
    text = re.sub(r"[^a-z\s.,!?;:'\-]", ' ', text)    
    text = re.sub(r'\s+', ' ', text).strip()
    return text

CORPUS_CLEAN = clean_text(CORPUS_RAW)

# Keras Tokenizer
# https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer
tokenizer = Tokenizer(
    num_words=None,
    oov_token='<OOV>',
    lower=True,
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)
tokenizer.fit_on_texts([CORPUS_CLEAN])

# Vocabulary statistics
word_index  = tokenizer.word_index
index_word  = tokenizer.index_word
VOCAB_SIZE  = len(word_index) + 1
TOTAL_WORDS = sum(tokenizer.word_counts.values())

print('Preprocessing')
print(f'  Raw corpus length   : {len(CORPUS_RAW):,} chars')
print(f'  Cleaned length      : {len(CORPUS_CLEAN):,} chars')
print(f'  Total word tokens   : {TOTAL_WORDS:,}')
print(f'  Vocabulary size     : {VOCAB_SIZE:,}   (words appearing ≥ 2 times)')
print(f'  OOV token idx       : {word_index["<OOV>"]}  (<OOV>)')
print(f'\nSample cleaned text (first 300 chars):\n', CORPUS_CLEAN[:300])
print(f'\nSample word index entries (first 10):', list(tokenizer.word_index.items())[:10])

---
## Cell 4 — Sequence Generation & Dataset Construction

In [ ]:
# Hyperparameters
SEQ_LEN = 40
STRIDE   = 3

# Convert entire corpus to a flat list of integer token Id
token_ids = tokenizer.texts_to_sequences([CORPUS_CLEAN])[0]

# Build (X, y) pairs using sliding window
X_seqs, y_seqs = [], []
for i in range(0, len(token_ids) - SEQ_LEN, STRIDE):
    X_seqs.append(token_ids[i : i + SEQ_LEN])
    y_seqs.append(token_ids[i + SEQ_LEN])

X_all = np.array(X_seqs, dtype=np.int32) 
y_all = np.array(y_seqs, dtype=np.int32)

# OHE
y_cat = to_categorical(y_all, num_classes=VOCAB_SIZE)

# ─── Train / Val / Test split  (80 / 10 / 10) ────────────────────────────────
# First split out 20% for val+test, then split that equally
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_all, y_cat, test_size=0.20, random_state=SEED, shuffle=True
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, shuffle=True
)

# Also keep integer labels for top-k accuracy evaluation later
_, y_int_tmp = train_test_split(y_all, test_size=0.20, random_state=SEED, shuffle=True)
y_val_int, y_test_int = train_test_split(y_int_tmp, test_size=0.50, random_state=SEED, shuffle=True)

print('── Dataset construction ──────────────────────────────────')
print(f'  Sequence length     : {SEQ_LEN}  tokens (input)')
print(f'  Sliding stride      : {STRIDE}   tokens')
print(f'  Total sequences     : {len(X_all):,}')
print(f'\n── Train / Val / Test split ──────────────────────────────')
print(f'  Training samples    : {len(X_train):,}  (80.0%)')
print(f'  Validation samples  : {len(X_val):,}   (10.0%)')
print(f'  Test samples        : {len(X_test):,}   (10.0%)')
print(f'\n  X_train shape       : {X_train.shape}')
print(f'  y_train shape       : {y_train.shape}')
print(f'  X_val shape         : {X_val.shape}')
print(f'  X_test shape        : {X_test.shape}')
print(f'\nMemory footprint (training X): {X_train.nbytes / 1e6:.1f} MB')

---
## Cell 5 — Model Architecture

In [ ]:
# https://www.geeksforgeeks.org/deep-learning/deep-learning-introduction-to-long-short-term-memory/
# https://www.tensorflow.org/text/tutorials/text_generation
# https://karpathy.github.io/2015/05/21/rnn-effectiveness/

---
## Cell 6 — Perplexity & Model Compilation

---
## Cell 7 — Training

---
## Cell 8 — Visualisations

---
## Cell 9 — Test Set Evaluation

---
## Cell 10 — Text Generation

---
## Cell 11 — Prediction 

---
## Cell 12 — Save Artefacts & Summary